# EuroSAT Few-Shot Benchmark

Compares four frozen pretrained backbones (ResNet-18, ConvNeXt-Tiny, ViT-B/16 and DINOv2 ViT-S/14) on fixed 4-way few-shot episodes from EuroSAT, with 1, 5 and 10 labeled images per class and optional augmentation of those labeled images.

`satellite image → (support augmentation) → frozen backbone → embedding → logistic regression → prediction`

**How to use**
1. Edit **Section 1 · Settings**. It is the only cell you need to change.
2. Run all cells in order. Start with `MODE = 'smoke'` to check that everything works.
3. Read the results in Sections 5 and 6.

| Section | What it does | Edit? |
|---|---|---|
| 1 · Settings | Hardware, run size, models, augmentation | **Yes** |
| 2 · Setup | Installs packages, connects storage, loads the project code | Only if your folders differ |
| 3 · Data check | Verifies the dataset and the fixed episodes | No |
| 4 · Run | Extracts embeddings and fits the classifiers (cached) | No |
| 5 · Results: models | Accuracy, comparison with ResNet-18, per-model details | No |
| 6 · Results: augmentation | Gains from support augmentation, if enabled | No |
| 7 · Saved files | Where results and figures are stored | No |

Smoke runs only check that the code runs; they are not research results. Background: `research_info.md` (plan), `figure_n_table_sug.md` (figures), `docs/` (methods).


## 1. Settings

- **Does:** chooses where to run, how much to run, which models, and which augmentations.
- **You get:** nothing yet. No data or models are loaded here.
- **Edit:** **this cell.** It is the only cell you need to change.


In [ ]:
# -- Where to run ---------------------------------------------------------------
TARGET = 'colab_gpu'   # 'colab_gpu' = Colab GPU runtime, 'local' = this computer
STORAGE_DIR = ''       # Shared folder for data, cache and results ('' = default, see Section 2)
CODE_DIR = ''          # Repository folder that contains src/ ('' = search automatically)
MANIFEST_FILE = ''     # Path to splits_manifest.json ('' = <storage>/splits_manifest.json)

# -- What to run ----------------------------------------------------------------
MODE = 'smoke'         # 'smoke' = 2 episodes per shot (quick check), 'full' = 600 test episodes per shot
MODELS = 'all'         # 'all', or a comma-separated subset: resnet18, convnext_tiny, vit_b_16, dinov2_vits14

# -- Augmentation of the labeled support images ---------------------------------
# 'none' (no augmentation) is required: every other condition is compared with it.
# 'repeat' copies each original (VIEWS + 1) times: same number of rows, no new pixels.
# Available: none, repeat, flip, rotation, translation, scale, brightness_contrast, noise, blur, erasing, cutout
POLICIES = ['none']
VIEWS = 4              # Augmented views added per original support image

# Strength of each method on the 224x224 image. Each view draws a new random value
# within these limits. Defaults: AUGMENTATION_DEFAULTS in src/eurosat/config.py.
AUGMENTATION = {
    # mirror left-right or top-bottom (50/50); no magnitude
    'flip': {},
    # quarter turn of 90, 180 or 270 degrees (lossless); no magnitude
    'rotation': {},
    # x and y shift ~ U(-f, +f) x 224 px each; empty border is black
    'translation': {'max_shift_fraction': 0.1},
    # zoom factor ~ U(1 - max_zoom, 1 + max_zoom)
    'scale': {'max_zoom': 0.2},
    # brightness and contrast factors ~ U(1 - max, 1 + max)
    'brightness_contrast': {'max_brightness': 0.3, 'max_contrast': 0.3},
    # add Gaussian noise with this std (pixels range 0-1)
    'noise': {'std': 0.05},
    # Gaussian blur; sigma ~ U(min_sigma, max_sigma) px; odd kernel_size
    'blur': {'kernel_size': 5, 'min_sigma': 0.1, 'max_sigma': 1.5},
    # square with side ~ U(min, max) x 224 px, filled with gray (fill)
    'erasing': {'min_size_fraction': 0.1, 'max_size_fraction': 0.3, 'fill': 0.5},
    # same square as erasing, filled with black (fill)
    'cutout': {'min_size_fraction': 0.1, 'max_size_fraction': 0.3, 'fill': 0.0},
}

# -- Display and report-only mode -----------------------------------------------
SHOW_DETAILS = False   # True = show every data-check table and extra result figures
REPORT_RESULTS = ''    # Path to a saved results.json: redraw results without data or GPU


## 2. Setup

- **Does:** installs missing packages, connects persistent storage (Google Drive on Colab), finds the project code, then checks the settings.
- **You get:** the folders in use and a summary of this run. Augmentation magnitudes are listed when augmentation is on (or with `SHOW_DETAILS`).
- **Edit:** nothing, unless your folders differ. On Colab, the shared Drive folder needs `splits_manifest.json` and the code in `code/src/` (or set `CODE_DIR` to a repository checkout). Restart the kernel after changing files in `src/`.


In [ ]:
import os
import sys
import subprocess
import importlib.util
from pathlib import Path

# Automated runs (scripts/execute_notebook.py) pass settings as EUROSAT_* environment variables.
TARGET = os.environ.get('EUROSAT_TARGET', TARGET)
STORAGE_DIR = os.environ.get('EUROSAT_STORAGE', STORAGE_DIR)
CODE_DIR = os.environ.get('EUROSAT_CODE_DIR', CODE_DIR)
if TARGET not in {'colab_gpu', 'local'}:
    raise ValueError("TARGET must be 'colab_gpu' or 'local'")

# 1) Install any missing packages.
requirements = {'torch': 'torch>=2.6,<3', 'torchvision': 'torchvision>=0.21,<1',
                'numpy': 'numpy>=1.26,<3', 'sklearn': 'scikit-learn>=1.4,<1.8',
                'PIL': 'Pillow>=10,<13', 'pandas': 'pandas>=2.2,<3',
                'matplotlib': 'matplotlib>=3.8,<4', 'seaborn': 'seaborn>=0.13,<0.14'}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing])

# 2) Persistent storage: Google Drive on Colab, the repository folder locally.
if TARGET == 'colab_gpu':
    from google.colab import drive
    drive.mount('/content/drive')
    parent = Path('/content/drive/MyDrive/ColabNotebooks/cs7643')
    default = parent / 'EuroSATLowData' if (parent / 'EuroSATLowData').exists() else parent / 'EuroSATRepresentationBenchmark'
    STORAGE = Path(STORAGE_DIR or default)
    if STORAGE_DIR and not STORAGE.is_dir():
        raise FileNotFoundError('Shared storage not found: check the My Drive shortcut and STORAGE_DIR')
    SCRATCH = Path('/content/eurosat-representation-benchmark')  # Extracted images on the fast local disk
else:
    here = Path.cwd()
    project = next((p for p in [here, *here.parents] if (p / 'pyproject.toml').exists()), here)
    STORAGE = Path(STORAGE_DIR or project)
    SCRATCH = STORAGE / 'data/extracted'
STORAGE.mkdir(parents=True, exist_ok=True)

# 3) Project code: a repository checkout, or <storage>/code/src on Colab.
candidates = ([Path(CODE_DIR)] if CODE_DIR else
              [Path.cwd(), *Path.cwd().parents, Path('/content/eurosat-representation-benchmark'), STORAGE / 'code'])
source_root = next((p for p in candidates if (p / 'src/eurosat/__init__.py').is_file()), None)
if source_root is None:
    raise FileNotFoundError('Project code not found: set CODE_DIR, or copy src/ into <storage>/code/src/')
sys.path.insert(0, str(source_root / 'src'))

print('Shared storage :', STORAGE)
print('Local images   :', SCRATCH)
print('Project code   :', source_root)


In [ ]:
from IPython.display import display
from eurosat import notebook as nb

config, MANIFEST, REPORT_PATH = nb.make_config(MODE, MODELS, POLICIES, VIEWS, AUGMENTATION,
                                               STORAGE, MANIFEST_FILE, REPORT_RESULTS)
REPORT_ONLY = bool(REPORT_PATH)
device = None if REPORT_ONLY else nb.check_device(TARGET)

display(nb.run_summary(config, TARGET, STORAGE, SCRATCH, MANIFEST, REPORT_ONLY))
if SHOW_DETAILS or config.policies != ['none']:
    display(nb.augmentation_settings(config))


## 3. Data check

- **Does:** verifies the dataset archive, checks every pool and episode in the shared split, then selects this run's fixed episodes. Nothing is resampled.
- **You get:** a pass/fail summary and one development support image per class. `SHOW_DETAILS = True` adds every table.
- **Edit:** nothing.


In [ ]:
if REPORT_ONLY:
    prepared = None
    print('Report-only mode: data check skipped.')
else:
    prepared = nb.check_data(config, STORAGE, SCRATCH, MANIFEST, show_details=SHOW_DETAILS)


## 4. Run the models

- **Does:** for each model in turn: extracts embeddings (reusing cached ones), picks the classifier's C on development episodes, then fits and scores every test episode. Backbones stay frozen.
- **You get:** progress logs and the results folder. Completed work is cached, so a rerun resumes.
- **Edit:** nothing. In report-only mode this loads the saved results instead.


In [ ]:
display(nb.model_table(config))
results = nb.run_or_load(config, TARGET, STORAGE, SCRATCH, MANIFEST, prepared, REPORT_PATH)


## 5. Results: model comparison

- **Does:** summarizes the no-augmentation results. Gains over ResNet-18 are paired on the same episodes and given in percentage points.
- **You get:** a summary table and accuracy by shot. `SHOW_DETAILS = True` adds more metrics, class recall and confusion matrices for all models.
- **Edit:** nothing.


In [ ]:
nb.show_model_overview(results, show_details=SHOW_DETAILS)


### Each model

Accuracy, Macro-F1, the selected C, class recall and a confusion matrix at the largest shot budget. Models left out of `MODELS` are marked as not run.


In [ ]:
nb.show_each_model(results)


## 6. Results: augmentation

- **Does:** compares each augmentation method with no augmentation on the same episodes.
- **You get:** the paired gain per method, the method chosen on development episodes for each model and shot (checked on test), and per-class effects. Empty when `POLICIES = ['none']`.
- **Edit:** nothing.


In [ ]:
nb.show_augmentation(results, config, show_details=SHOW_DETAILS)


## 7. Saved files

- **Does:** lists where this run's outputs are stored.
- **You get:** `results.json`, summary tables, figures (PNG and PDF) and data-check exports.
- **Edit:** nothing.


In [ ]:
display(nb.output_files(results, prepared))
